# Dose Analysis - Multi-File Comparison

This notebook loads multiple CSV files (representing different doses) and compares absorbance at specific wavelengths over time.

## Overview
- Load multiple processed spectral data files
- Extract absorbance at specified wavelengths
- Compare dose responses
- Visualize all doses on a single plot

## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os

# Enable inline plotting
%matplotlib inline

## Load Multiple CSV Files

Update the file path pattern below to match your data files:

In [ ]:
# Update this path pattern to match your data files
# Example: file_pattern = 'path/to/your/final_pyspec_*.csv'
file_pattern = 'final_pyspec_*.csv'  # Default: look in current directory

# Load multiple CSV files
file_paths = glob.glob(file_pattern)
data_frames = [pd.read_csv(file, index_col=0) for file in file_paths]

print(f"Number of files loaded: {len(file_paths)}")
print("\nFile paths:")
for i, path in enumerate(file_paths, 1):
    print(f"  {i}. {os.path.basename(path)}")

## Configuration

Set the wavelengths to analyze and time cutoff:

In [ ]:
# Configuration
selected_wavelengths = [412]  # List of wavelengths to extract (nm)
cut_timepoint = 20  # Time cutoff (seconds)

print(f"Selected wavelengths: {selected_wavelengths} nm")
print(f"Time cutoff: {cut_timepoint} s")

## Extract Data at Specified Wavelengths

In [ ]:
# Extract specified wavelengths from all files
extracted_data = []
dose_labels = []

for file_path, df in zip(file_paths, data_frames):
    wavelengths = df.index.values.astype(float)
    time = df.columns.str.replace('s', '').astype(float)
    
    # Extract dose label from filename
    dose_label = os.path.basename(file_path).split('_')[-1].split('.')[0]
    dose_labels.append(dose_label)
    
    extracted_row = []
    for selected_wavelength in selected_wavelengths:
        # Find closest wavelength
        closest_wavelength = min(wavelengths, key=lambda x: abs(x - selected_wavelength))
        
        if closest_wavelength in wavelengths:
            absorbance = df.loc[closest_wavelength].values
            
            # Ensure time and absorbance arrays are of the same length
            if len(time) == len(absorbance):
                # Cut the data at the specified time point
                mask = time <= cut_timepoint
                time_cut = time[mask]
                absorbance_cut = absorbance[mask]
                
                extracted_row.append(absorbance_cut)
    
    extracted_data.append(extracted_row)

print(f"\nExtracted data from {len(extracted_data)} files")
print(f"Dose labels: {dose_labels}")

## Save Extracted Data

Save the extracted wavelength data to a CSV file:

In [ ]:
# Write extracted data to a new CSV file
# Note: This creates a simple format; you may want to adjust based on your needs
extracted_df = pd.DataFrame(
    extracted_data, 
    columns=[f'{wavelength} nm' for wavelength in selected_wavelengths]
)
extracted_df.insert(0, 'Dose', dose_labels)
extracted_df.to_csv('extracted_wavelengths.csv', index=False)

print("Extracted data saved to 'extracted_wavelengths.csv'")
print("\nExtracted data preview:")
display(extracted_df)

## Plot All Doses Together

Create a comprehensive plot showing all doses for comparison:

In [ ]:
# Plot the data for each dose individually
plt.figure(figsize=(12, 8))

for i, (df, file_path, dose_label) in enumerate(zip(data_frames, file_paths, dose_labels)):
    wavelengths = df.index.values.astype(float)
    time = df.columns.str.replace('s', '').astype(float)
    
    for selected_wavelength in selected_wavelengths:
        # Find closest wavelength
        closest_wavelength = min(wavelengths, key=lambda x: abs(x - selected_wavelength))
        
        if closest_wavelength in wavelengths:
            absorbance = df.loc[closest_wavelength].values
            
            # Ensure time and absorbance arrays are of the same length
            if len(time) == len(absorbance):
                # Cut the data at the specified time point
                mask = time <= cut_timepoint
                time_cut = time[mask]
                absorbance_cut = absorbance[mask]
                
                # Plot the data
                plt.scatter(time_cut, absorbance_cut, 
                          label=f'Dose {dose_label} @ {closest_wavelength}nm', 
                          s=10, alpha=0.6)

# Finalize the plot
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel('Time (s)', fontsize=12)
plt.ylabel('Absorbance', fontsize=12)
plt.title('Absorbance vs Time for Different Doses', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Individual Dose Plots

Create separate subplots for each dose:

In [ ]:
# Create subplots for each dose
n_files = len(data_frames)
n_cols = min(3, n_files)
n_rows = (n_files + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
if n_files == 1:
    axes = [axes]
else:
    axes = axes.flatten()

for idx, (df, file_path, dose_label) in enumerate(zip(data_frames, file_paths, dose_labels)):
    wavelengths = df.index.values.astype(float)
    time = df.columns.str.replace('s', '').astype(float)
    
    ax = axes[idx]
    
    for selected_wavelength in selected_wavelengths:
        # Find closest wavelength
        closest_wavelength = min(wavelengths, key=lambda x: abs(x - selected_wavelength))
        
        if closest_wavelength in wavelengths:
            absorbance = df.loc[closest_wavelength].values
            
            # Ensure time and absorbance arrays are of the same length
            if len(time) == len(absorbance):
                # Cut the data at the specified time point
                mask = time <= cut_timepoint
                time_cut = time[mask]
                absorbance_cut = absorbance[mask]
                
                # Plot the data
                ax.scatter(time_cut, absorbance_cut, s=10, alpha=0.6)
                ax.set_title(f'Dose {dose_label} @ {closest_wavelength}nm')
                ax.set_xlabel('Time (s)')
                ax.set_ylabel('Absorbance')
                ax.grid(True, alpha=0.3)

# Hide unused subplots
for idx in range(n_files, len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

## Summary Statistics

Calculate and display summary statistics for each dose:

In [ ]:
# Calculate summary statistics
summary_data = []

for df, dose_label in zip(data_frames, dose_labels):
    wavelengths = df.index.values.astype(float)
    time = df.columns.str.replace('s', '').astype(float)
    
    for selected_wavelength in selected_wavelengths:
        closest_wavelength = min(wavelengths, key=lambda x: abs(x - selected_wavelength))
        
        if closest_wavelength in wavelengths:
            absorbance = df.loc[closest_wavelength].values
            
            if len(time) == len(absorbance):
                mask = time <= cut_timepoint
                absorbance_cut = absorbance[mask]
                
                summary_data.append({
                    'Dose': dose_label,
                    'Wavelength': closest_wavelength,
                    'Mean Abs': absorbance_cut.mean(),
                    'Std Abs': absorbance_cut.std(),
                    'Max Abs': absorbance_cut.max(),
                    'Min Abs': absorbance_cut.min(),
                    'Initial Abs': absorbance_cut[0] if len(absorbance_cut) > 0 else np.nan,
                    'Final Abs': absorbance_cut[-1] if len(absorbance_cut) > 0 else np.nan
                })

summary_df = pd.DataFrame(summary_data)
print("\nSummary Statistics:")
display(summary_df)

## Save Final Plot

Save the comparison plot to a file:

In [ ]:
# Recreate and save the final plot
plt.figure(figsize=(12, 8))

for i, (df, file_path, dose_label) in enumerate(zip(data_frames, file_paths, dose_labels)):
    wavelengths = df.index.values.astype(float)
    time = df.columns.str.replace('s', '').astype(float)
    
    for selected_wavelength in selected_wavelengths:
        closest_wavelength = min(wavelengths, key=lambda x: abs(x - selected_wavelength))
        
        if closest_wavelength in wavelengths:
            absorbance = df.loc[closest_wavelength].values
            
            if len(time) == len(absorbance):
                mask = time <= cut_timepoint
                time_cut = time[mask]
                absorbance_cut = absorbance[mask]
                
                plt.scatter(time_cut, absorbance_cut, 
                          label=f'Dose {dose_label} @ {closest_wavelength}nm', 
                          s=10, alpha=0.6)

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel('Time (s)', fontsize=12)
plt.ylabel('Absorbance', fontsize=12)
plt.title('Absorbance vs Time for Different Doses', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('all_doses_comparison.png', dpi=300, bbox_inches='tight')
plt.close()

print("Plot saved to 'all_doses_comparison.png'")